## Substitutes

In [ ]:
import pandas as pd
%run ../utils/substitutes.py

file_path_root = "../data/validation/substitutes/"

# Load files
sampled_df = pd.read_csv('../data/results/sample-pairwise.csv')
product_df = pd.read_csv('../data/cleaned/product-info-full.csv')
sampled_products = pd.read_csv('../data/results/sampled-products.csv')
orders_full_df = pd.read_csv('../dataset/order_products__prior.csv')

Calculate a hybrid substitution score for the sampled products and save to CSV.

In [7]:
compute_sub_score_by_dept(
    product_df,
    sampled_df,
    sampled_products['product_id'].tolist(),
    f"{file_path_root}scored-subs.csv")

Completed substitute calculations. Saved to ../data/validation/substitutes/scored-subs.csv


Filter the substitutes from the previous step based on calculated threshold and mark as identified substitutes. Save to CSV.

In [8]:
substitutes_df = pd.read_csv(f"{file_path_root}scored-subs.csv")

# Find the best substitution score threshold which will identify a product as a true substitute
best_threshold = find_best_threshold(substitutes_df)

# Add a new column 'identified_substitute' based on the threshold
substitutes_df['identified_substitute'] = substitutes_df['score'] >= best_threshold

# Save results to CSV
substitutes_df.to_csv(f"{file_path_root}final-subs.csv", index=False)

 Best threshold determined as: 0.598 and adjusted to: 0.6157535762735594



Calculate a transferability % for the identified substitutes. Save to CSV.

In [ ]:
results = compute_transferability(orders_full_df, substitutes_df, top_n=10)
results.to_csv(f"{file_path_root}transfer.csv", index=False)

Validate the results from the previous steps.

In [11]:
transfer_df = pd.read_csv(f'{file_path_root}transfer.csv')

orders_df = orders_full_df[['order_id', 'product_id']]

results = run_multiple_subsets_validation(
    orders_df=orders_df,
    subs_df=substitutes_df,
    transfer_df=transfer_df, 
    n_subsets=5,
    sample_size=100,
    method="freq_stratified",   # "random" or "freq_stratified"
    p_switch=0.25,
    K=3,
    n_trials=20,
    seed=2025
)

print(results)
print("Aggregated means:")
print(results.mean(numeric_only=True))

   subset_index  n_sample_products  stability_mean_corr  stability_std_corr  \
0             0                100             0.991367            0.009731   
1             1                100             0.990564            0.009301   
2             2                100             0.988105            0.014467   
3             3                100             0.991049            0.010583   
4             4                100             0.990867            0.012326   

   effectiveness_precision_mean  effectiveness_precision_std  eff_runtime_s  \
0                      0.720167                     0.166419       0.650000   
1                      0.721167                     0.147745       0.630000   
2                      0.731833                     0.168809       0.506003   
3                      0.721667                     0.139071       0.632996   
4                      0.723167                     0.156847       0.610998   

   eff_mem_growth_mb  
0           1.386719  
1   